# TD 2 — Prise en main de Linopy et PyPSA

**Objectif :** construire et résoudre des problèmes d’optimisation linéaire appliqués aux systèmes électriques, d’abord avec **Linopy** (formulation explicite du problème), puis avec **PyPSA** (description d’un réseau électrique dont PyPSA construit le modèle d’optimisation).

Le TD est organisé en deux parties :
1. **Linopy** : des variables et contraintes simples jusqu’à deux zones de marché reliées par une interconnexion.
2. **PyPSA** : construction d’un réseau à deux nœuds, optimisation du dispatch, lecture des résultats et contrainte globale d’émissions.

Les cellules de solution sont balisées par `### DEBUT SOLUTION` et `### FIN SOLUTION` pour faciliter la préparation de la version étudiant.


## Préparation de l’environnement

Si nécessaire, installez les dépendances dans l’environnement Python du TD :

```sh
pip install linopy highspy pypsa matplotlib
```

`highspy` fournit le solveur HiGHS utilisé dans les exemples. Selon votre environnement, certaines fonctions de visualisation interactives peuvent nécessiter des dépendances supplémentaires.


# Partie 1 — Linopy

Linopy permet de formuler un problème d’optimisation à l’aide de variables, de contraintes et d’une fonction objectif. On commence par un problème jouet, puis on ajoute des dimensions et on termine par un marché électrique constitué de deux zones reliées par une ligne.

:::{dropdown} YouTube Video
:::{iframe} https://www.youtube.com/embed/a0IAUh6Q63k
:width: 100%
Video not loading? Click [here](https://www.youtube.com/watch?v=a0IAUh6Q63k).
:::

:::{note}
This material is in part adapted from the following resources:
- [Linopy Getting Started](https://linopy.readthedocs.io/en/latest/index.html)
- [PyPSA simple electricity market examples](https://docs.pypsa.org/latest/examples/simple-electricity-market-examples/)
:::

<img src="https://github.com/PyPSA/linopy/blob/master/doc/logo.png?raw=true" width="300px" />

[Linopy](https://linopy.readthedocs.io/en/latest/index.html)  is an open-source framework for formulating, solving, and analyzing optimization problems with Python.

With Linopy, you can create optimization models within Python that consist of decision variables, constraints, and optimization objectives. You can then solve these instances using a variety of commercial and open-source solvers (specialised software).

[Linopy](https://linopy.readthedocs.io/en/latest/index.html) supports a wide range of problem types, including:

- **Linear programming**
- Integer programming
- Mixed-integer programming
- Quadratic programming


:::{note}
Documentation for this package is available at https://linopy.readthedocs.io.
:::

## Solve a Basic Model

In this example, we explain the basic functions of the linopy `Model` class. First, we are setting up a very simple linear optimization model, given by

Minimize:
    $$x + 2y$$
subject to:
    $$ x \ge 0 $$
    $$y \ge 0 $$
    $$3x + 7y \ge 10 $$
    $$5x + 2y \ge 3 $$

### Initializing a `Model`

The Model class in Linopy is a fundamental part of the library. It serves as a container for all the relevant data associated with a linear optimization problem. This includes variables, constraints, and the objective function.

In [ ]:
import linopy

m = linopy.Model()

This creates a new Model object, which you can then use to define your optimization problem.

:::{note}
It is good practice to choose a short variable name (like `m`) to reduce the verbosity of your code.
:::

### Adding decision variables

**Variables** are the unknowns of an optimisation problems and are intended to be given values by solving an optimisation problem. A variable can always be assigned with a lower and an upper bound. In our case, both `x` and `y` have a lower bound of zero (default is unbouded). In linopy, you can add variables to a `Model` using the `add_variables()` method:

In [ ]:
x = m.add_variables(lower=0, name="x")
y = m.add_variables(lower=0, name="y");

`x` and `y` are linopy variables of the class `linopy.Variable`. Each of them contain all relevant information that define it. The `name` parameter is optional but can be useful for referencing the variables later.

In [ ]:
x

In [ ]:
m.variables

In [ ]:
m.variables["x"]

### Adding Constraints

**Constraints** are equality or inequality expressions that define the *feasible* space of the decision variables. They consist of the left hand side (LHS) and the right hand side (RHS). The first constraint that we want to write down is $3x + 7y = 10$ which we write out exactly in the mathematical way:

In [ ]:
3 * x + 7 * y >= 10

Note, we can also mix the constant and the variable expression, like this

In [ ]:
3 * x + 7 * y - 10 >= 0

… and linopy will automatically take over the separation of variables expression on the LHS, and constant values on the RHS.

The constraint is currently not assigned to the model. We assign it by calling the `add_constraints()` function:

In [ ]:
m.add_constraints(3 * x + 7 * y >= 10)
m.add_constraints(5 * x + 2 * y >= 3);

In [ ]:
m.constraints

In [ ]:
m.constraints["con0"]

### Adding the Objective 

The objective function defines what you want to optimize. It is a function of variables that a solver attempts to maximize or minimize. You can set the objective function of a `linopy.Model` using the `add_objective()` method. For our example that would be

In [ ]:
m.add_objective(x + 2 * y, sense="min")

In [ ]:
m.objective

Note, we can either minimize or maximize in linopy. Per default, linopy applies `sense='min'` so it is not necessary to explicitly define the optimization sense. In summary:

In [ ]:
m

### Solving the Model

Once you've defined your `linopy.Model`  with variables, constraints, and an objective function, you can solve it using the `solve` method:

In [ ]:
m.solve()

Solvers are needed to compute solutions to the optimization models. There exists a large variety of solvers. In many cases, they specialise in certain problem types or solving algorithms, e.g. linear or nonlinear problems.

- **open-source examples**: [HiGHS](https://highs.dev), [SCIP](https://www.scipopt.org/), [CBC](https://www.coin-or.org/Cbc/), [GLPK](https://www.gnu.org/software/glpk/), [Ipopt](https://coin-or.github.io/Ipopt/)
- **commercial examples**: [Gurobi](https://www.gurobi.com/), [COPT](https://www.shanshu.ai/copt) [CPLEX](https://www.ibm.com/de-de/analytics/cplex-optimizer), [FICO Xpress](https://www.fico.com/en/products/fico-xpress-optimization)

The open-source solvers are sufficient to handle meaningful linopy models with hundreds to several thousand variables and constraints. However, as applications get large or more complex, there may be a need to turn to a commercial solvers (which often provide free academic licenses).

For this course, we use HiGHS, which is already in the course environment `esm-ss-26`.

### Retrieving optimisation results

The solution of the linear problem is assigned to the variables under `solution` in form of a `xarray.Dataset`.

In [ ]:
x.solution

In [ ]:
y.solution

We can also read out the objective value:

In [ ]:
m.objective.value

And the dual values (or shadow prices) of the model's constraints: 

In [ ]:
m.dual["con0"]

Well done! You solved your first linopy model!

## Use Coordinates

Now, the real power of the package comes into play! 

Linopy is structured around the concept that variables, and therefore expressions and constraints, have coordinates. That is, a `Variable` object actually contains multiple variables across dimensions, just as we know it from a `numpy` array or a `pandas.DataFrame`.

Suppose the two variables `x` and `y` are now functions of time `t` and we would modify the problem according to: 

Minimize:
$$\sum_t x_t + 2 y_t$$

subject to:

$$x_t \ge 0 \qquad \forall t $$
$$y_t \ge 0 \qquad \forall t $$
$$3x_t + 7y_t \ge 10 t \qquad \forall t$$
$$5x_t + 2y_t \ge 3 t \qquad \forall t$$

whereas `t` spans all the range from 0 to 10.

In order to formulate the new problem with linopy, we start again by initializing a model.

In [ ]:
m = linopy.Model()

Again, we define `x` and `y` using the `add_variables()` function, but now we are adding a `coords` argument. This automatically creates optimization variables for all coordinates, in this case time-steps `t`.

In [ ]:
import pandas as pd

time = pd.Index(range(10), name="time")

x = m.add_variables(
    lower=0,
    coords=[time],
    name="x",
)
y = m.add_variables(lower=0, coords=[time], name="y")

In [ ]:
x

Following the previous example, we write the constraints out using the syntax from above, while multiplying the RHS with `t`. Note that the coordinates from the LHS and the RSH have to match. 

:::{note}
In the beginning, it is recommended to use explicit dimension names. In this way, things remain clear and no unexpected broadcasting (which we show later) will happen.
:::

In [ ]:
factor = pd.Series(time, index=time)

3 * x + 7 * y >= 10 * factor

It always helps to write out the constraints before adding them to the model. Since they look good, let's assign them.

In [ ]:
con1 = m.add_constraints(3 * x + 7 * y >= 10 * factor, name="con1")
con2 = m.add_constraints(5 * x + 2 * y >= 3 * factor, name="con2")
m

Now, when it comes to the objective, we use the `sum` function of `linopy.LinearExpression`. This stacks all terms all terms of the `time` dimension and writes them into one big expression. 

In [ ]:
obj = (x + 2 * y).sum()

In [ ]:
obj

In [ ]:
m.add_objective(obj, overwrite=True)

Then, we can solve:

In [ ]:
m.solve()

In order to inspect the solution. You can go via the variables, i.e. `y.solution` or via the `solution` aggregator of the model, which combines the solution of all variables.

In [ ]:
m.solution.to_dataframe()

Sometimes it can be helpful to plot the solution:

In [ ]:
m.solution.to_dataframe().plot(grid=True, ylabel="Optimal Value");

Alright! Now you learned how to set up linopy variables and expressions with coordinates. For more advanced `linopy` operations you can check out the [User Guide](https://linopy.readthedocs.io/en/latest/user-guide.html).

## Electricity Market Examples

### Single bidding zone, single period

We want to minimise operational cost of an example electricity system representing South Africa subject to generator limits and meeting the load:

\begin{equation}
    \min_{g_s} \sum_s o_s g_s
  \end{equation}
  such that
  \begin{align}
    g_s &\leq G_s \\
    g_s &\geq 0 \\
    \sum_s g_s &= d
  \end{align}

We are given the following information on the South African electricity system:

Marginal costs in EUR/MWh

In [ ]:
marginal_costs = pd.Series([0, 30, 60, 80], index=["Wind", "Coal", "Gas", "Oil"])
marginal_costs

Power plant capacities in MW

In [ ]:
capacities = pd.Series([3000, 35000, 8000, 2000], index=["Wind", "Coal", "Gas", "Oil"])
capacities

Inelastic demand in MW

In [ ]:
load = 42000

We now start building the model

In [ ]:
m = linopy.Model()

Let's define the dispatch variables `g` with the `lower` and `upper` bound:
$$g_s \leq G_s $$
$$g_s \geq 0 $$

In [ ]:
g = m.add_variables(lower=0, upper=capacities, coords=[capacities.index], name="g")
g

And and the objective to minimize total operational costs:
$$\min_{g_s} \sum_s o_s g_s$$

In [ ]:
m.add_objective(marginal_costs.values * g, sense="min")
m.objective

Which is subject to: 

$$\sum_s g_s = d$$

In [ ]:
m.add_constraints(g.sum() == load, name="energy_balance")

Then, we can solve the model:

In [ ]:
m.solve()

This is the optimimal generator dispatch (MW)

In [ ]:
m.solution.to_dataframe()

And the market clearing price we can read from the shadow price of the energy balance constraint (i.e. the added cost of increasing electricity demand by one unit):

In [ ]:
m.dual["energy_balance"]

### Two bidding zones with transmission

Let's add a spatial dimension, such that the optimisation problem is expanded to
\begin{equation}
  \min_{g_{i,s}, f_\ell} \sum_s o_{i,s} g_{i,s}
\end{equation}
such that
\begin{align}
  g_{i,s} &\leq G_{i,s} \\
  g_{i,s} &\geq 0 \\
  \sum_s g_{i,s} - \sum_\ell K_{i\ell} f_\ell &= d_i & \text{KCL} \\
  |f_\ell| &\leq F_\ell & \text{line limits}  \\
  \sum_\ell C_{\ell c} x_\ell f_\ell &= 0 & \text{KVL} 
\end{align}

In this example, we connect the previous South African electricity system with a hydro generation unit in Mozambique through a single transmission line. Note that because a single transmission line will not result in any cycles, we can neglect KVL in this case.

We are given the following data (all in MW):

In [ ]:
generators = ["Coal", "Wind", "Gas", "Oil", "Hydro"]
countries = ["South_Africa", "Mozambique"]

In [ ]:
capacities = pd.DataFrame(
    {
        "Coal": [35000, 0],
        "Wind": [3000, 0],
        "Gas": [8000, 0],
        "Oil": [2000, 0],
        "Hydro": [0, 1200],
    },
    index=countries,
)
capacities.index.name = "countries"
capacities.columns.name = "generators"

capacities

In [ ]:
# variable costs in EUR/MWh
marginal_costs = pd.Series([30, 0, 60, 80, 0], index=generators)
marginal_costs.index.name = "generators"
marginal_costs

In [ ]:
load = pd.Series([42000, 650], index=countries)
load.index.name = "countries"
load

In [ ]:
transmission = 500

Let's start with a new model instance

In [ ]:
m = linopy.Model()

Now we create dispatch variables, as before, with the `upper` and `lower` bound for each countries and generators.

In [ ]:
capacities

In [ ]:
g = m.add_variables(lower=0, upper=capacities, name="g")
g

We now define the line limit for the transmission line, assuming that power flowing from Mozambique	to South Africa is positive.

The line limit equation can be defined as   
\begin{align}
|f_\ell| &\leq F_\ell & \text{line limits}
\end{align}

In [ ]:
f = m.add_variables(lower=-transmission, upper=transmission, name="flow_MZ_SA")
f

The energy balance constraint is replaced by KCL, where we take into account local generation as well as incoming or outgoing flows. The KCL equation can be defined as:
\begin{align}
  \sum_s g_{i,s} - \sum_\ell K_{i\ell} f_\ell &= d_i & \text{KCL} \\
\end{align}

We also need the incidence matrix $K_{i\ell}$ of this network (here it's very simple!) and assume some direction for the flow variable. Here, we picked the orientation from South Africa to Mozambique. This means that if the values for the flow variable $f_\ell$ are positive South Africa exports to Mozambique and vice versa if the variable takes negative values.

In [ ]:
for country in countries:
    sign = -1 if country == "Mozambique" else 1  # minimal incidence matrix
    m.add_constraints(
        g.loc[country].sum() + sign * f == load[country],
        name=f"{country}_KCL",
    )

In [ ]:
m.constraints["Mozambique_KCL"]

The objective can be written as:
$$\min_{g_{i,s}, f_\ell} \sum_s o_{i,s} g_{i,s}$$

In [ ]:
obj = (g * marginal_costs).sum()
obj

In [ ]:
m.add_objective(obj, sense="min")

We now solve the model.

In [ ]:
m.solve()

Now, we print the optimization results

In [ ]:
m.objective.value

In [ ]:
g.solution.to_dataframe()

In [ ]:
m.constraints["South_Africa_KCL"].dual

In [ ]:
m.constraints["Mozambique_KCL"].dual

### Exercice 1 — Sensibilité à la capacité d’interconnexion

Reprenez le modèle à deux zones construit ci-dessus. Étudiez l’effet d’une capacité de transport de **0, 250, 500, 1 000 et 2 000 MW**.

1. Pour chaque capacité, calculez le coût total du système.
2. Relevez le flux sur l’interconnexion.
3. Comparez les prix d’équilibre (valeurs duales des bilans nodaux) des deux zones.
4. Expliquez à partir de quelle capacité les prix des deux zones deviennent identiques, ou pourquoi ils restent différents.

Complétez la boucle ci-dessous. Le modèle doit être reconstruit à chaque itération afin de ne pas réutiliser les contraintes du modèle précédent.

In [ ]:
# TODO : tester plusieurs capacités de transport et stocker les résultats
### DEBUT SOLUTION
results = []

for transmission_capacity in [0, 250, 500, 1000, 2000]:
    m_sens = linopy.Model()

    g_sens = m_sens.add_variables(
        lower=0,
        upper=capacities,
        name="g",
    )
    f_sens = m_sens.add_variables(
        lower=-transmission_capacity,
        upper=transmission_capacity,
        name="flow_MZ_SA",
    )

    for country in countries:
        sign = -1 if country == "Mozambique" else 1
        m_sens.add_constraints(
            g_sens.loc[country].sum() + sign * f_sens == load[country],
            name=f"{country}_KCL",
        )

    m_sens.add_objective((g_sens * marginal_costs).sum(), sense="min")
    m_sens.solve(solver_name="highs")

    results.append({
        "capacity_MW": transmission_capacity,
        "cost_EUR_per_hour": float(m_sens.objective.value),
        "flow_MW": float(f_sens.solution),
        "price_South_Africa": float(m_sens.constraints["South_Africa_KCL"].dual),
        "price_Mozambique": float(m_sens.constraints["Mozambique_KCL"].dual),
    })

results_df = pd.DataFrame(results).set_index("capacity_MW")
results_df
### FIN SOLUTION


**À retenir.** Avec Linopy, vous écrivez directement les variables de décision, les contraintes physiques et la fonction objectif. Cela rend la formulation transparente, mais implique de gérer explicitement les bilans et les limites du réseau.

# Partie 2 — PyPSA

PyPSA propose une représentation structurée d’un système énergétique : un réseau (`Network`) rassemble des nœuds (`Bus`), des générateurs (`Generator`), des consommations (`Load`) et des éléments de transport (`Line`). À partir de ces données, `n.optimize()` construit et résout le problème d’optimisation.

Nous reprenons un exemple à deux zones — Afrique du Sud et Mozambique — et nous nous arrêtons avant l’exemple plus réaliste utilisant le réseau SciGRID.

<img src="https://docs.pypsa.org/latest/assets/logo/logo-primary-light.svg#only-light" width="300px" />

:::{dropdown} YouTube Video
:::{iframe} https://www.youtube.com/embed/ORZgewOo1Ro
:width: 100%
Video not loading? Click [here](https://www.youtube.com/watch?v=ORZgewOo1Ro).
:::

PyPSA stands for **Python for Power System Analysis**.

PyPSA is an open source Python package for simulating and optimising modern energy systems that include features such as

- conventional generators with unit commitment (ramp-up, ramp-down, start-up, shut-down),
- time-varying wind and solar generation,
- energy storage with efficiency losses and inflow/spillage for hydroelectricity
- coupling to other energy sectors (electricity, transport, heat, industry),
- conversion between energy carriers (e.g. electricity to hydrogen),
- transmission networks (AC, DC, other fuels)

PyPSA can be used for a variety of problem types (e.g. electricity market modelling, long-term investment planning, transmission network expansion planning), and is designed to scale well with large networks and long time series.

Compared to building power system by hand in `linopy`, PyPSA does the following things for you:

- manage data inputs in standardised format
- build standardised optimisation problem
- communicate with the solver(s)
- retrieve and process optimisation results
- manage data outputs in standardised format

The [documentation](https://docs.pypsa.org/) describes it as follows: 

> PyPSA is an open-source Python framework for optimising and simulating modern power and energy systems that include features such as conventional generators with unit commitment, variable wind and solar generation, hydro-electricity, inter-temporal storage, coupling to other energy sectors, elastic demands, and linearised power flow with loss approximations in DC and AC networks. PyPSA is designed to scale well with large networks and long time series. It is made for researchers, planners and utilities with basic coding aptitude who need a fast, easy-to-use and transparent tool for power and energy system analysis.

### Key Dependencies

- `pandas` for storing data about network components and time series
- `numpy` and `scipy` for linear algebra and sparse matrix calculations
- `matplotlib` and `cartopy` for plotting on a map
- `networkx` for network calculations
- `linopy` for handling optimisation problems

:::{note}
Documentation for this package is available at https://docs.pypsa.org.
:::

## Basic Structure

| Component | Description |
| --- | --- |
| [Network](https://docs.pypsa.org/latest/user-guide/design/#network-components) | Container for all components. |
| [Bus](https://docs.pypsa.org/latest/user-guide/components/buses/) | Node where components attach. |
| [Carrier](https://docs.pypsa.org/latest/user-guide/components/carriers/) | Energy carrier or technology (e.g. electricity, hydrogen, gas, coal, oil, biomass, on-/offshore wind, solar). Can track properties such as specific carbon dioxide emissions or nice names and colors for plots. |
| [Load](https://docs.pypsa.org/latest/user-guide/components/loads/) | Energy consumer (e.g. electricity demand). |
| [Generator](https://docs.pypsa.org/latest/user-guide/components/generators/) | Generator (e.g. power plant, wind turbine, PV panel). |
| [Line](https://docs.pypsa.org/latest/user-guide/components/lines/) | Power distribution and transmission lines (overhead and cables). |
| [Link](https://docs.pypsa.org/latest/user-guide/components/links/) | Links connect two buses with controllable energy flow, direction-control and losses. They can be used to model: <ul><li>HVDC links</li><li>HVAC li'nes (neglecting KVL, only net transfer capacities (NTCs))</li><li>conversion between carriers (e.g. electricity to hydrogen in electrolysis)</li></ul> |
| [StorageUnit](https://docs.pypsa.org/latest/user-guide/components/storage-units/) | Storage with fixed nominal energy-to-power ratio. |
| [GlobalConstraint](https://docs.pypsa.org/latest/user-guide/components/global-constraints/) | Constraints affecting many components at once, such as emission limits. |
| [Store](https://docs.pypsa.org/latest/user-guide/components/stores/) | Storage with separately extendable energy capacity. |


:::{note}
Links in the table lead to documentation for each component.
:::

<img src="https://docs.pypsa.org/latest/assets/images/buses.png" width="500px" />


:::{warning}
Per unit values of voltage and impedance are used internally for network calculations. It is assumed internally that the base power is **1 MW**.
:::

## From structured data to optimisation

**The design principle of PyPSA is that each component is associated with a set of variables and constraints that will be added to the optimisation model based on the input data stored for the components.**

For an *hourly* electricity market simulation, PyPSA will solve an optimisation problem that looks like this

\begin{equation}
\min_{g_{i,s,t}; f_{\ell,t}; g_{i,r,t,\text{charge}}; g_{i,r,t,\text{discharge}}; e_{i,r,t}} \sum_s o_{s} g_{i,s,t}
\end{equation}
such that
\begin{align}
0 & \leq g_{i,s,t} \leq \hat{g}_{i,s,t} G_{i,s}  & \text{generation limits : generator} \\
-F_\ell &\leq f_{\ell,t} \leq F_\ell & \text{transmission limits : line}  \\
d_{i,t} &= \sum_s g_{i,s,t} + \sum_r g_{i,r,t,\text{discharge}} - \sum_r g_{i,r,t,\text{charge}} - \sum_\ell K_{i\ell} f_{\ell,t} & \text{KCL : bus} \\
 0 &=\sum_\ell C_{\ell c} x_\ell f_{\ell,t} & \text{KVL : cycles} \\
0 & \leq g_{i,r,t,\text{discharge}} \leq G_{i,r,\text{discharge}}& \text{discharge limits : storage unit} \\
    0 & \leq g_{i,r,t,\text{charge}} \leq G_{i,r,\text{charge}} & \text{charge limits : storage unit} \\
    0 & \leq e_{i,r,t} \leq E_{i,r} & \text{energy limits : storage unit} \\ 
    e_{i,r,t} &= \eta^0_{i,r,t} e_{i,r,t-1} + \eta^1_{i,r,t}g_{i,r,t,\text{charge}} -  \frac{1}{\eta^2_{i,r,t}} g_{i,r,t,\text{discharge}} & \text{consistency : storage unit} \\
    e_{i,r,0} & = e_{i,r,|T|-1}  & \text{cyclicity : storage unit}
\end{align}

**Decision variables:**

- $g_{i,s,t}$ is the generator dispatch at bus $i$, technology $s$, time step $t$,
- $f_{\ell,t}$ is the power flow in line $\ell$,
- $g_{i,r,t,\text{dis-/charge}}$ denotes the charge and discharge of storage unit $r$ at bus $i$ and time step $t$,
- $e_{i,r,t}$ is the state of charge of storage $r$ at bus $i$ and time step $t$.

**Parameters:**

- $o_{i,s}$ is the marginal generation cost of technology $s$ at bus $i$,
- $x_\ell$ is the reactance of transmission line $\ell$,
- $K_{i\ell}$ is the incidence matrix,
- $C_{\ell c}$ is the cycle matrix,
- $G_{i,s}$ is the nominal capacity of the generator of technology $s$ at bus $i$,
- $F_{\ell}$ is the rating of the transmission line $\ell$,
- $E_{i,r}$ is the energy capacity of storage $r$ at bus $i$,
- $\eta^{0/1/2}_{i,r,t}$ denote the standing (0), charging (1), and discharging (2) efficiencies.

:::{note}
For a full reference to the optimisation problem description, see https://docs.pypsa.org/latest/user-guide/optimization/overview/
:::

## Simple electricity market example

Let's get acquainted with PyPSA to build a variant of one of the simple electricity market models we previously built in `linopy`.

We have the following data:

- fuel costs in € / MWh$_{th}$ 

In [ ]:
fuel_cost = dict(
    coal=8,
    gas=100,
    oil=48,
)

- efficiencies of thermal power plants in MWh$_{el}$ / MWh$_{th}$

In [ ]:
efficiency = dict(
    coal=0.33,
    gas=0.58,
    oil=0.35,
)

- specific emissions in t$_{CO_2}$ / MWh$_{th}$

In [ ]:
# t/MWh thermal
emissions = dict(
    coal=0.34,
    gas=0.2,
    oil=0.26,
    hydro=0,
    wind=0,
)

- power plant capacities in MW

In [ ]:
power_plants = {
    "SA": {"coal": 35000, "wind": 3000, "gas": 8000, "oil": 2000},
    "MZ": {"hydro": 1200},
}

- electrical load in MW

In [ ]:
loads = {
    "SA": 42000,
    "MZ": 650,
}

## Building a basic network

By convention, PyPSA is imported without an alias:

In [ ]:
import pypsa

In [ ]:
pypsa.options.params.optimize.include_objective_constant = True

First, we create a new network object which serves as the overall container for all components.

In [ ]:
n = pypsa.Network()

The second component we need are buses. **Buses** are the fundamental nodes of the network, to which all other components like loads, generators and transmission lines attach. They enforce energy conservation for all elements feeding in and out of it (i.e. Kirchhoff’s Current Law).

<img src="https://docs.pypsa.org/latest/assets/images/buses.png" width="500px" />

Components can be added to the network `n` using the `n.add()` function. It takes the component name as a first argument, the name of the component as a second argument and possibly further parameters as keyword arguments. Let's use this function, to add buses for each country to our network:

In [ ]:
n.add("Bus", "SA", y=-30.5, x=25, v_nom=400)
n.add("Bus", "MZ", y=-18.5, x=35.5, v_nom=400)

For each class of components, the data describing the components is stored in a `pandas.DataFrame`. For example, all static data for buses is stored in `n.buses`

In [ ]:
n.buses

You see there are many more attributes than we specified while adding the buses; many of them are filled with default parameters which were added. You can look up the field description, defaults and status (required input, optional input, output) for buses here https://docs.pypsa.org/latest/user-guide/components/buses/, and analogously for all other components. 

The method `n.add()` also allows you to add multiple components at once. For instance, multiple **carriers** for the fuels with information on specific carbon dioxide emissions, a nice name, and colors for plotting. For this, the function takes the component name as the first argument and then a list of component names and then optional arguments for the parameters. Here, scalar values, lists, dictionary or `pandas.Series` are allowed. The latter two needs keys or indices with the component names.

In [ ]:
n.add(
    "Carrier",
    ["coal", "gas", "oil", "hydro", "wind"],
    co2_emissions=emissions,
    nice_name=["Coal", "Gas", "Oil", "Hydro", "Onshore Wind"],
    color=["dimgrey", "tomato", "olive", "seagreen", "royalblue"],
)

n.add("Carrier", "AC", nice_name="Electricity", color="crimson")

The `n.add()` function is very general. It lets you add any component to the network object `n`. For instance, in the next step we add **generators** for all the different power plants.

In Mozambique:

In [ ]:
n.add(
    "Generator",
    "MZ hydro",
    bus="MZ",
    carrier="hydro",
    p_nom=1200,  # MW
    marginal_cost=0,  # default
)

In South Africa (in a loop):

In [ ]:
for tech, p_nom in power_plants["SA"].items():
    n.add(
        "Generator",
        f"SA {tech}",
        bus="SA",
        carrier=tech,
        efficiency=efficiency.get(tech, 1),
        p_nom=p_nom,
        marginal_cost=fuel_cost.get(tech, 0) / efficiency.get(tech, 1),
    )

As a result, the `n.generators` DataFrame looks like this:

In [ ]:
n.generators

Next, we're going to add the electricity demand. This time, without a loop.

A positive value for `p_set` means consumption of power from the bus (in MW).

In [ ]:
n.add(
    "Load",
    "SA electricity demand",
    bus="SA",
    p_set=loads["SA"],
    carrier="AC",
)

In [ ]:
n.add(
    "Load",
    "MZ electricity demand",
    bus="MZ",
    p_set=loads["MZ"],
    carrier="AC",
)

In [ ]:
n.loads

Finally, we add the connection between Mozambique and South Africa with a 500 MW line:

In [ ]:
n.add(
    "Line",
    "SA-MZ",
    bus0="SA",
    bus1="MZ",
    s_nom=500,
    x=1,
    r=1,
)

In [ ]:
n.lines

We can have a sneak peek at the network we built with the `n.plot()` function.

In [ ]:
n.plot(bus_size=1, margin=1);

Or with its interactive sibling `n.explore()`.

In [ ]:
n.explore(bus_size=2e3, line_width=10)

## Optimisation

With all input data transferred into PyPSA's data structure, we can now build and run the resulting optimisation problem. In PyPSA, building, solving and retrieving results from the optimisation model is contained in a single function call `n.optimize()`. This function optimizes the operational dispatch decisions for least cost, as well as investment decisions for components marked as extendable (more on this next time).

The `n.optimize()` function can take many arguments. The most relevant for the moment is the choice of the solver. We already know some solvers from the introduction to `linopy` (e.g. "highs" and "gurobi"). They need to be installed in your environment (e.g. `conda`), to use them here:

In [ ]:
n.optimize(solver_name="highs", log_to_console=False)

Let's have a look at the results.

Since the power flow and dispatch are generally time-varying quantities, these are stored in a different location than e.g. `n.generators`. They are stored in `n.generators_t`. Thus, to find out the dispatch of the generators, run

In [ ]:
n.generators_t.p

or if you prefer it in relation to the generators nominal capacity

In [ ]:
n.generators_t.p / n.generators.p_nom

You see that the time index has the value 'now'. This is the default index when no time series data has been specified and the network only covers a single state (e.g. a particular hour). 

Similarly you will find the power flow in transmission lines at

In [ ]:
n.lines_t.p0

In [ ]:
n.lines_t.p1

The `p0` will tell you the flow from `bus0` to `bus1`. `p1` will tell you the flow from `bus1` to `bus0`.

What about the shadow prices?

In [ ]:
n.buses_t.marginal_price

## Explore the optimisation model

Under the hood, a call to `n.optimize()` builds a `linopy` optimisation model, solves it with the specified solver, and then retrieves the results back into the PyPSA data structure. We can access this intermediate `linopy` model via `n.model`. This allows us to explore the optimisation model in more detail, for instance to see all variables and constraints that were added to the optimisation problem.

In [ ]:
n.model

In [ ]:
n.model.constraints["Generator-fix-p-upper"]

In [ ]:
n.model.constraints["Bus-nodal_balance"]

In [ ]:
n.model.objective

In [ ]:
n.model.constraints["Generator-fix-p-upper"].dual.to_dataframe()

## Basic network plotting

The results can also be visualised with PyPSA's built-in plotting functions. The `n.plot()` function uses `matplotlib` to create static plots, while `n.explore()` creates interactive plots based on `pydeck`. In the following, we will focus on the interactive plotting with `n.explore()`.

The `n.explore()` function has a variety of styling arguments to tweak the appearance of the buses, the lines and the map in the background. For example, we can size the buses according to their load:

In [ ]:
s = n.loads.groupby("bus").p_set.sum()
s

In [ ]:
n.explore(
    bus_size=s,
    bus_color="orange",
    bus_alpha=1,
    line_color="orchid",
    line_width=30,
)

The `bus_size` argument of `n.explore()` can be even more powerful. 

The dispatch of each generator, we can find at:

In [ ]:
n.generators_t.p.loc["now"]

If we group this by the **bus** and **carrier** and use it as `bus_size`, we can create pie charts at each bus showing the generation mix.

In [ ]:
s = n.generators_t.p.loc["now"].groupby([n.generators.bus, n.generators.carrier]).sum()
s

Additionally, we can show the line flows by specifying the `line_flow` argument.

In [ ]:
n.explore(
    bus_size=s,
    line_width=30,
    line_flow=n.lines_t.p0.loc["now"] / 5,
)

In the backend, the plotting function will look up the colors specified in `n.carriers` for each carrier and match it with the second index-level of `s`.

## Modifying networks

Modifying data of components in an existing PyPSA network is as easy as modifying the entries of a `pandas.DataFrame`. For instance, if we want to reduce the cross-border transmission capacity between South Africa and Mozambique, we'd run:

In [ ]:
n.lines.loc["SA-MZ", "s_nom"] = 400
n.lines

In [ ]:
n.optimize(log_to_console=False)

You can see that the production of the hydro power plant was reduced and that of the gas power plant increased owing to the reduced transmission capacity.

In [ ]:
n.generators_t.p

## Global constraints for emission limits

In the example above, we happen to have some spare gas capacity with lower carbon intensity than the coal and oil generators. We could use this to lower the emissions of the system, but it will be more expensive. We can implement the limit of carbon dioxide emissions as a constraint.

This is achieved in PyPSA through **Global Constraints** which add constraints that apply to many components at once.

But first, we need to calculate the current level of emissions to set a sensible limit.

We can compute the emissions per generator (in tonnes of CO$_2$) in the following way.

$$\frac{g_{i,s,t} \cdot \rho_{i,s}}{\eta_{i,s}}$$

where $ \rho$ is the specific emissions (tonnes/MWh thermal) and $\eta$ is the conversion efficiency (MWh electric / MWh thermal) of the generator with dispatch $g$ (MWh electric):

In [ ]:
e = (
    n.generators_t.p
    / n.generators.efficiency
    * n.generators.carrier.map(n.carriers.co2_emissions)
)
e

Summed up, we get total emissions in tonnes:

In [ ]:
e.sum().sum()

So, let's say we want to reduce emissions by 10%:

In [ ]:
n.add(
    "GlobalConstraint",
    "emission_limit",
    carrier_attribute="co2_emissions",
    sense="<=",
    constant=e.sum().sum() * 0.9,
)

In [ ]:
n.optimize(log_to_console=False)

In [ ]:
n.generators_t.p

In [ ]:
n.generators_t.p / n.generators.p_nom

In [ ]:
n.global_constraints.mu

Can we lower emissions even further? Say by another 5% points?

In [ ]:
n.global_constraints.loc["emission_limit", "constant"] = 0.85

In [ ]:
n.optimize(log_to_console=False)

**No!** Without any additional capacities, we have exhausted our options to reduce emissions in that hour. The solver tells us that the problem is *infeasible*, i.e. there is no solution that satisfies all constraints. Better revert this change:

In [ ]:
n.global_constraints.loc["emission_limit", "constant"] = 0.9

## Data import and export

:::{note}
Documentation: https://docs.pypsa.org/latest/user-guide/import-export/.
:::

You may find yourself in a need to store PyPSA networks for later use. Or, maybe you want to import the genius PyPSA example that someone else uploaded to the web to explore.

Among other file formats, PyPSA can be stored as `netCDF` (`.nc`) file or as an Excel file (`.xlsx`). The approach is similar for all formats:

In [ ]:
n.export_to_netcdf("tmp.nc");

In [ ]:
n_nc = pypsa.Network("tmp.nc")

## Exercice 2 — Modifier le réseau PyPSA

Utilisez le réseau `n` défini et optimisé dans la partie précédente. Pour chaque question, créez une copie de `n` afin de conserver le réseau de référence.

**Question 1 — Perte de l’interconnexion.** Supprimez la ligne `SA-MZ`. Le système reste-t-il faisable ? Comparez le dispatch et le coût opérationnel avec le cas de référence. Expliquez le rôle de la centrale hydroélectrique du Mozambique.

**Question 2 — Augmentation de la capacité éolienne.** Doublez la capacité nominale du générateur `SA wind`. Comment le prix marginal de l’électricité en Afrique du Sud évolue-t-il ? Quel générateur fixe le prix ?

**Question 3 — Nouvelle centrale nucléaire.** Ajoutez en Afrique du Sud une centrale nucléaire de 1 000 MW avec un coût marginal de 20 €/MWh. Comparez le prix marginal et le dispatch avant et après l’ajout.

Avant de commencer, on retire la contrainte globale d’émissions introduite dans le tutoriel : les trois questions portent sur le réseau sans cette limite.

In [ ]:
# TODO : retirer la contrainte globale d'émissions si elle existe
### DEBUT SOLUTION
if "emission_limit" in n.global_constraints.index:
    n.remove("GlobalConstraint", "emission_limit")
# Réinitialiser le modèle de solveur éventuellement conservé après les optimisations précédentes
n.model.solver_model = None
### FIN SOLUTION


In [ ]:
# TODO : question 1 — retirer la ligne et résoudre le modèle
### DEBUT SOLUTION
n_t1 = n.copy()
n_t1.remove("Line", "SA-MZ")
n_t1.optimize(solver_name="highs", log_to_console=False)

print("Dispatch sans interconnexion :")
display(n_t1.generators_t.p)
print("Bilan énergétique :")
display(n_t1.statistics.energy_balance().astype(int))
### FIN SOLUTION


In [ ]:
# TODO : question 2 — doubler la capacité éolienne et analyser le prix
### DEBUT SOLUTION
n_t2 = n.copy()
n_t2.generators.loc["SA wind", "p_nom"] *= 2
n_t2.optimize(solver_name="highs", log_to_console=False)

print("Prix marginaux :")
display(n_t2.buses_t.marginal_price)

price_sa = n_t2.buses_t.marginal_price.at["now", "SA"]
print("Prix en Afrique du Sud :", price_sa, "€/MWh")
print(
    "Générateur dont le coût marginal correspond au prix :",
    n_t2.generators.index[
        (n_t2.generators.bus == "SA")
        & (n_t2.generators.marginal_cost.round(2) == round(price_sa, 2))
    ].tolist(),
)
### FIN SOLUTION


In [ ]:
# TODO : question 3 — ajouter une centrale nucléaire et comparer les prix
### DEBUT SOLUTION
n_t3 = n.copy()
if "nuclear" not in n_t3.carriers.index:
    n_t3.add("Carrier", "nuclear", co2_emissions=0, nice_name="Nuclear", color="purple")

n_t3.add(
    "Generator",
    "SA nuclear",
    bus="SA",
    carrier="nuclear",
    p_nom=1000,
    marginal_cost=20,
)
n_t3.optimize(solver_name="highs", log_to_console=False)

print("Prix marginaux après ajout du nucléaire :")
display(n_t3.buses_t.marginal_price)
print("Dispatch après ajout du nucléaire :")
display(n_t3.generators_t.p)
### FIN SOLUTION


## Bilan du TD

- **Linopy** : le modèle d’optimisation est écrit explicitement ; les variables, les contraintes, l’objectif et les valeurs duales sont directement accessibles.
- **PyPSA** : le système est décrit à l’aide de composants physiques ; PyPSA construit le modèle d’optimisation sous-jacent et restitue les résultats dans des tableaux dédiés.
- Dans les deux approches, les capacités de production, les contraintes de transport et les bilans énergétiques déterminent le dispatch et les prix marginaux.
